<a href="https://colab.research.google.com/github/ek7anna/AI-TASKS/blob/main/chunking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install langchain langchain-community
!pip -q install langchain-text-splitters
!pip -q install langchain-experimental
!pip -q install langchain-huggingface
!pip -q install sentence-transformers
!pip -q install pypdf

In [ ]:
from google.colab import files

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_experimental.text_splitter import SemanticChunker

from langchain_huggingface import HuggingFaceEmbeddings

import pandas as pd

/tmp/ipykernel_525/710379997.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/tmp/ipykernel_525/710379997.py:7: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


In [ ]:
uploaded = files.upload()

pdf_file = list(uploaded.keys())[0]

print("Uploaded File:", pdf_file)

Saving HSE Dashboard Project Plan.pdf to HSE Dashboard Project Plan.pdf
Uploaded File: HSE Dashboard Project Plan.pdf


In [ ]:
loader = PyPDFLoader(pdf_file)

pages = loader.load()

print("Total Pages:", len(pages))

Total Pages: 15


In [ ]:
document_text = ""

for page in pages:
    document_text += page.page_content + "\n"

print("Total Characters:", len(document_text))

print("\nFirst 500 Characters:\n")

print(document_text[:500])

Total Characters: 6656

First 500 Characters:

HSE DASHBOARD PROJECT PLAN
Project Overview
Build a simple dynamic HSE Dashboard using:
Backend:
Python
FastAPI
PostgreSQL
Frontend:
HTML
CSS
JavaScript
Bootstrap (optional)
The screenshots are only for reference.
The main goal is:
Create records
Store data in PostgreSQL
Display live data in Dashboard
Dashboard updates automatically from database
PROJECT MODULES
We will build only 5 modules:
Authentication & Users
Incidents
Tasks
Training
Dashboard
These modules are enough to build the dashboard


In [ ]:
# -----------------------------
# Recursive Chunking
# -----------------------------

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=75,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

recursive_chunks = recursive_splitter.create_documents([document_text])

print("Recursive Chunk Count :", len(recursive_chunks))

print("\nFirst Chunk\n")

print(recursive_chunks[0].page_content)

Recursive Chunk Count : 13

First Chunk

HSE DASHBOARD PROJECT PLAN
Project Overview
Build a simple dynamic HSE Dashboard using:
Backend:
Python
FastAPI
PostgreSQL
Frontend:
HTML
CSS
JavaScript
Bootstrap (optional)
The screenshots are only for reference.
The main goal is:
Create records
Store data in PostgreSQL
Display live data in Dashboard
Dashboard updates automatically from database
PROJECT MODULES
We will build only 5 modules:
Authentication & Users
Incidents
Tasks
Training
Dashboard
These modules are enough to build the dashboard shown in the screenshots.
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
1. 
2. 
3. 
4. 
5. 
1


In [ ]:
lengths = [len(chunk.page_content) for chunk in recursive_chunks]

print("="*50)

print("Recursive Chunking Statistics")

print("="*50)

print("Total Chunks :", len(lengths))

print("Average Length :", round(sum(lengths)/len(lengths),2))

print("Largest Chunk :", max(lengths))

print("Smallest Chunk :", min(lengths))

Recursive Chunking Statistics
Total Chunks : 13
Average Length : 567.15
Largest Chunk : 598
Smallest Chunk : 308


In [ ]:
# =====================================
# Hierarchical Chunking
# =====================================

# Parent splitter
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1800,
    chunk_overlap=150
)

parent_chunks = parent_splitter.create_documents([document_text])

# Child splitter
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=75
)

child_chunks = []

for parent in parent_chunks:
    child_chunks.extend(
        child_splitter.create_documents([parent.page_content])
    )

print("Parent Chunks :", len(parent_chunks))
print("Child Chunks  :", len(child_chunks))

print("\nSample Child Chunk\n")
print(child_chunks[0].page_content)

Parent Chunks : 4
Child Chunks  : 19

Sample Child Chunk

HSE DASHBOARD PROJECT PLAN
Project Overview
Build a simple dynamic HSE Dashboard using:
Backend:
Python
FastAPI
PostgreSQL
Frontend:
HTML
CSS
JavaScript
Bootstrap (optional)
The screenshots are only for reference.
The main goal is:
Create records
Store data in PostgreSQL
Display live data in Dashboard
Dashboard updates automatically from database
PROJECT MODULES
We will build only 5 modules:
Authentication & Users
Incidents
Tasks
Training
Dashboard


In [ ]:
child_lengths = [len(chunk.page_content) for chunk in child_chunks]

print("=" * 50)
print("Hierarchical Chunking Statistics")
print("=" * 50)

print("Parent Chunks :", len(parent_chunks))
print("Child Chunks  :", len(child_chunks))

print("Average Length :", round(sum(child_lengths) / len(child_lengths), 2))
print("Largest Chunk  :", max(child_lengths))
print("Smallest Chunk :", min(child_lengths))

Hierarchical Chunking Statistics
Parent Chunks : 4
Child Chunks  : 19
Average Length : 426.05
Largest Chunk  : 496
Smallest Chunk : 91


In [ ]:
# =====================================
# Semantic Chunking
# =====================================

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

semantic_splitter = SemanticChunker(
    embeddings=embedding_model,
    breakpoint_threshold_type="percentile"
)

semantic_chunks = semantic_splitter.create_documents([document_text])

print("Semantic Chunk Count :", len(semantic_chunks))

print("\nFirst Semantic Chunk\n")

print(semantic_chunks[0].page_content)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Semantic Chunk Count : 3

First Semantic Chunk

HSE DASHBOARD PROJECT PLAN
Project Overview
Build a simple dynamic HSE Dashboard using:
Backend:
Python
FastAPI
PostgreSQL
Frontend:
HTML
CSS
JavaScript
Bootstrap (optional)
The screenshots are only for reference. The main goal is:
Create records
Store data in PostgreSQL
Display live data in Dashboard
Dashboard updates automatically from database
PROJECT MODULES
We will build only 5 modules:
Authentication & Users
Incidents
Tasks
Training
Dashboard
These modules are enough to build the dashboard shown in the screenshots. • 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
1.


In [ ]:
semantic_lengths = [len(chunk.page_content) for chunk in semantic_chunks]

print("=" * 50)
print("Semantic Chunking Statistics")
print("=" * 50)

print("Total Chunks :", len(semantic_chunks))
print("Average Length :", round(sum(semantic_lengths) / len(semantic_lengths), 2))
print("Largest Chunk :", max(semantic_lengths))
print("Smallest Chunk :", min(semantic_lengths))

Semantic Chunking Statistics
Total Chunks : 3
Average Length : 2213.67
Largest Chunk : 3907
Smallest Chunk : 562


In [ ]:
comparison = pd.DataFrame({
    "Strategy": [
        "Recursive",
        "Hierarchical",
        "Semantic"
    ],
    "Chunks": [
        len(recursive_chunks),
        len(child_chunks),
        len(semantic_chunks)
    ],
    "Average Length": [
        round(sum(lengths)/len(lengths),2),
        round(sum(child_lengths)/len(child_lengths),2),
        round(sum(semantic_lengths)/len(semantic_lengths),2)
    ],
    "Largest Chunk": [
        max(lengths),
        max(child_lengths),
        max(semantic_lengths)
    ],
    "Smallest Chunk": [
        min(lengths),
        min(child_lengths),
        min(semantic_lengths)
    ]
})

comparison

,Strategy,Chunks,Average Length,Largest Chunk,Smallest Chunk
0,Recursive,13,567.15,598,308
1,Hierarchical,19,426.05,496,91
2,Semantic,3,2213.67,3907,562
